# H SC 790 · Lab 12 · Did the Rollout Work?

**Week 13.** Due Thursday, April 29, 11:59 PM · 30 points.

**What earns full credit**
- Runs end to end from a fresh runtime
- Naive, adjusted, and controlled estimates compared
- Balancing measures reported
- AI-use note included

**Rules.** Everything here is synthetic. Never load protected health information or confidential employer data into Colab or GitHub. The lab is finished when it runs top to bottom from a fresh runtime (Runtime → Restart session and run all).

The data dictionary for this lab is the README in the lab's folder of the course repository.

Cells marked `# YOUR CODE HERE` and *Your answer here.* are yours to complete. Add cells freely.

In [ ]:
# Setup: run this cell first, every time.
import os, json, urllib.request
import numpy as np
import pandas as pd

# The course data, read straight from the course repository. Never from your own machine.
COURSE_URL = os.environ.get("HSC790_DATA", "https://raw.githubusercontent.com/lmlevine/hsc790-spr27/main/data/")

def read_text(rel):
    """Text of one file in the course data folder, for example read_text('SNAPSHOT.txt')."""
    return urllib.request.urlopen(COURSE_URL + rel).read().decode()

print("Data snapshot:", read_text("SNAPSHOT.txt").strip())

Puente County phased in electronic case reporting (eCR) in place of faxed reports entered by hand. Wave 1 facilities went live in week 105 (the week of June 30, 2025). Wave 2 facilities had not started by week 156. Did the rollout work, and should the county expand it, pause, or stop?

In [ ]:
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson

weekly = pd.read_csv(COURSE_URL + "lab12/weekly.csv", parse_dates=["week_start"])
val = pd.read_csv(COURSE_URL + "lab12/timestamp_validation.csv", parse_dates=["fax_received_ts", "entered_ts"])
T0 = 105                                 # go-live week
weekly.head(3)

## 1. Plot the series and do the naive comparison

Mark go-live. Compute the before-and-after difference in means for Wave 1. Keep it to compare with later answers.

In [ ]:
# YOUR CODE HERE

## 2. Fit the segmented regression

Time, post, and time since go-live. Report the level change and the slope change with intervals. Start without the seasonal terms and robust errors, so you can see in step 3 what they change.

In [ ]:
w1 = weekly[weekly.wave == 1].copy()
w1['post']  = (w1.week >= T0).astype(int)
w1['since'] = (w1.week - T0).clip(lower=0)

# YOUR CODE HERE

## 3. Fix the errors and the seasons

Check the residuals for autocorrelation and for a seasonal pattern. Refit with Newey–West (HAC) errors and month terms: the slide's model. What changed? `maxlags` is a judgment; try more than one value.

In [ ]:
# YOUR CODE HERE

In [ ]:
m = smf.ols(
    'median_days ~ week + post + since + C(month)',
    data=w1,
).fit(cov_type='HAC', cov_kwds={'maxlags': 4})

print(m.params[['post', 'since']])       # level, slope
print(m.conf_int().loc[['post', 'since']])

# counterfactual: the earlier trend, continued
w1['cf'] = m.predict(w1.assign(post=0, since=0))
effect_26 = (w1.median_days - w1.cf)[w1.week == T0 + 26]

w1.plot(x='week', y=['median_days', 'cf'])

In [ ]:
# YOUR CODE HERE

## 4. Handle phase-in and measurement

Look at `pct_electronic` in the weeks after go-live, and model or exclude the ramp-up. Then look at the validation file: what does it tell you about how "received" was recorded for faxed reports, and what does that do to the outcome when reports become electronic? Adjust for it and refit.

In [ ]:
# YOUR CODE HERE

## 5. Add the comparison series

Fit Wave 2 over the same period. Did anything happen to Wave 2 around the time Wave 1 went live? Then fit both waves together so that Wave 2 absorbs whatever affected everyone. What does the difference between the waves tell you?

In [ ]:
# YOUR CODE HERE

## 6. Check balancing measures and recommend

Duplicates, queue length, and occupation completeness, in Wave 1 before and after go-live, against Wave 2 over the same weeks. Then write the recommendation in the last interpretation question.

In [ ]:
# YOUR CODE HERE

## Interpretation

Cite your own output in each answer.

**1. How wrong was the naive answer?** Compare it with the adjusted and controlled estimates, and explain the gap.

*Your answer here.*

**2. How much is measurement?** What share of the apparent gain comes from the change in time-stamping?

*Your answer here.*

**3. What got worse?** Report each balancing measure and who carries the cost.

*Your answer here.*

**4. Expand, pause, or stop?** Your recommendation, its conditions, and who must agree to it.

*Your answer here.*

## AI-use note (required)

Say which AI tools you used, for what, and one thing you checked or corrected in their output. "None" is an acceptable answer.

*Your note here.*

## Submit

1. Runtime → Restart session and run all. Fix anything that fails.
2. File → Save a copy in GitHub, into `labs/lab12/` of your course repository.